In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, glob, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - calibration metrics on the CALIBRATED probabilities, evaluated on the
# SOURCE calibration pool (held out from the isotonic fit set D_probcal, and
# in-distribution, so calibration SHOULD hold). Per class one-vs-rest Brier and
# ECE (equal-mass bins, matching nb04's estimator), plus top-label ECE, and
# reliability-curve points. Raw pre-calibration probs were not cached, so this
# reports the calibrated quality that the conformal layer actually consumes.
# =============================================================================
NBINS=15
def brier_ovr(P,y,ncls):
    return {c: float(np.mean((P[:,c]-(y==c).astype(float))**2)) for c in range(ncls)}
def ece_equal_mass_1d(p,correct,nbins=NBINS):
    if len(p)<nbins: nbins=max(1,len(p)//2)
    o=np.argsort(p); p,correct=p[o],correct[o]
    bins=np.array_split(np.arange(len(p)),nbins)
    return float(sum(len(b)/len(p)*abs(correct[b].mean()-p[b].mean()) for b in bins if len(b)))
def ece_ovr(P,y,ncls,nbins=NBINS):
    return {c: ece_equal_mass_1d(P[:,c],(y==c).astype(float),nbins) for c in range(ncls)}
def ece_toplabel(P,y,nbins=NBINS):
    conf=P.max(1); correct=(P.argmax(1)==y).astype(float); return ece_equal_mass_1d(conf,correct,nbins)
def reliability_pts(P,y,c,nbins=10):
    p=P[:,c]; t=(y==c).astype(float); o=np.argsort(p); p,t=p[o],t[o]
    bins=np.array_split(np.arange(len(p)),nbins); out=[]
    for b in bins:
        if len(b): out.append((float(p[b].mean()),float(t[b].mean()),int(len(b))))
    return out
print('calibration metric helpers ready; equal-mass bins =',NBINS)


calibration metric helpers ready; equal-mass bins = 15


In [3]:
# =============================================================================
# Cell 3 - NSL calibration on S_pool (calibrated array in probs_*.npz), 30 models.
# =============================================================================
CLASSES=config.CANONICAL_CLASSES; c2i={c:i for i,c in enumerate(CLASSES)}; K=len(CLASSES)
nsl_train=pd.read_parquet(config.INTERIM_DIR/'nslkdd_train.parquet').reset_index(drop=True)
part=pd.read_parquet(config.PROC_DIR/'nslkdd_source_partition_labels.parquet')
nsl_train=nsl_train.assign(partition=part['partition'].values)
y_sp_nsl=nsl_train[nsl_train.partition=='source_cal_pool']['label'].map(c2i).to_numpy()

rows=[]; toplab=[]
for f in sorted(glob.glob(str(config.PROC_DIR/'probs_*.npz'))):
    arch,seed=Path(f).stem.replace('probs_','').rsplit('_s',1)
    P=np.load(f)['S_pool'].astype(np.float64)
    assert len(P)==len(y_sp_nsl), 'NSL S_pool length mismatch'
    br=brier_ovr(P,y_sp_nsl,K); ec=ece_ovr(P,y_sp_nsl,K)
    for c in range(K):
        rows.append({'dataset':'nslkdd','arch':arch,'seed':seed,'class':CLASSES[c],
                     'brier':br[c],'ece':ec[c]})
    toplab.append({'dataset':'nslkdd','arch':arch,'seed':seed,'toplabel_ece':ece_toplabel(P,y_sp_nsl)})
nsl_cal=pd.DataFrame(rows)
print('NSL calibration (mean over 30 models, per class):')
print(nsl_cal.groupby('class').agg(brier=('brier','mean'),ece=('ece','mean')).round(4).to_string())
print('NSL top-label ECE:', round(float(pd.DataFrame(toplab)['toplabel_ece'].mean()),4))


NSL calibration (mean over 30 models, per class):
         brier     ece
class                 
DoS     0.0003  0.0002
Normal  0.0018  0.0005
Probe   0.0007  0.0001
R2L     0.0007  0.0001
U2R     0.0002  0.0001
NSL top-label ECE: 0.0006


In [4]:
# =============================================================================
# Cell 4 - CIC + UGR calibration on their source calibration pools (calibrated
# srcpool arrays), evaluated with reconstructed labels.
# =============================================================================
rows=[]; toplab=[]
# ---- CIC: 5 realizations x 30 models; srcpool labels per realization ----
cic=pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed=cic[cic['day']=='wednesday'].reset_index(drop=True); wed=wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
CICC=['Benign','DoS'];
def lab_cic(idx): return (wed.loc[idx,'label'].to_numpy()=='DoS').astype(int)
REAL=['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
      'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']
for name in REAL:
    spx=np.load(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy'); ysp=lab_cic(spx)
    for f in sorted((config.DATA_DIR/'cic_probs').glob(f'{name}__*.npz')):
        _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
        P=np.load(f)['srcpool'].astype(np.float64)
        if len(P)!=len(ysp): continue
        br=brier_ovr(P,ysp,2); ec=ece_ovr(P,ysp,2)
        for c in range(2): rows.append({'dataset':'cicids2017','arch':arch,'seed':seed,'realization':name,
                                        'class':CICC[c],'brier':br[c],'ece':ec[c]})
        toplab.append({'dataset':'cicids2017','toplabel_ece':ece_toplabel(P,ysp)})
# ---- UGR: 30 models; source pool labels ----
UGR=config.DATASETS_DIR/'ugr16'; usrc=pd.read_parquet(UGR/'july_week5.parquet')
usrc['label']=usrc['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']; usrc=usrc[usrc.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}
def strat(df,fr,seed,col='label'):
    rng=np.random.default_rng(seed); nm=list(fr); ff=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(ff))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(ff*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
usrc=usrc.assign(partition=strat(usrc,config.SPLIT_FRACTIONS,20260725).values)
y_sp_ugr=usrc[usrc.partition=='source_cal_pool']['label'].map(U2I).to_numpy()
for f in sorted((config.DATA_DIR/'ugr16_probs').glob('ugr16__*.npz')):
    _,arch,sd=Path(f).stem.split('__'); seed=int(sd.replace('seed',''))
    P=np.load(f)['srcpool'].astype(np.float64)
    if len(P)!=len(y_sp_ugr): continue
    br=brier_ovr(P,y_sp_ugr,len(UCL)); ec=ece_ovr(P,y_sp_ugr,len(UCL))
    for c in range(len(UCL)): rows.append({'dataset':'ugr16','arch':arch,'seed':seed,'realization':'july',
                                           'class':UCL[c],'brier':br[c],'ece':ec[c]})
    toplab.append({'dataset':'ugr16','toplabel_ece':ece_toplabel(P,y_sp_ugr)})
cu_cal=pd.DataFrame(rows); cu_top=pd.DataFrame(toplab)
print('CIC calibration (mean, per class):')
print(cu_cal[cu_cal.dataset=='cicids2017'].groupby('class').agg(brier=('brier','mean'),ece=('ece','mean')).round(4).to_string())
print('\nUGR calibration (mean, per class):')
print(cu_cal[cu_cal.dataset=='ugr16'].groupby('class').agg(brier=('brier','mean'),ece=('ece','mean')).round(4).to_string())
print('\ntop-label ECE:', cu_top.groupby('dataset')['toplabel_ece'].mean().round(4).to_dict())


CIC calibration (mean, per class):
         brier     ece
class                 
Benign  0.0001  0.0001
DoS     0.0001  0.0001

UGR calibration (mean, per class):
              brier     ece
class                      
background   0.0158  0.0014
dos          0.0001  0.0001
nerisbotnet  0.0144  0.0011
scan11       0.0171  0.0019
scan44       0.0173  0.0014

top-label ECE: {'cicids2017': 0.0001, 'ugr16': 0.0024}


In [ ]:
# =============================================================================
# Cell 5 - consolidate table + reliability curves + verdict + commit.
# =============================================================================
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
cal=pd.concat([nsl_cal[['dataset','arch','seed','class','brier','ece']],
               cu_cal[['dataset','arch','seed','class','brier','ece']]],ignore_index=True)
summary=cal.groupby(['dataset','class']).agg(brier=('brier','mean'),ece=('ece','mean')).round(4).reset_index()
summary.to_csv(config.REPORTS_DIR/'calibration_quality.csv',index=False)
print('CALIBRATION QUALITY (calibrated probs, source pool, per class):')
print(summary.to_string(index=False))

# overall per-dataset means
overall=cal.groupby('dataset').agg(mean_brier=('brier','mean'),mean_ece=('ece','mean')).round(4)
print('\nper-dataset mean Brier / ECE:'); print(overall.to_string())

verdict={'analysis':'calibration quality of the calibrated probabilities feeding the conformal layer',
   'evaluated_on':'source calibration pool (held out from isotonic fit set, in-distribution)',
   'per_dataset_mean':overall.reset_index().to_dict('records'),
   'note':('raw pre-calibration probs were not cached, so a before/after comparison would need retraining '
           '(out of scope). Low ECE/Brier here shows the isotonic-calibrated probabilities are well '
           'calibrated in-distribution, so the coverage failures under shift are NOT a base-calibration '
           'artifact. ECE estimator: equal-mass bins (nb04 convention).')}
(config.REPORTS_DIR/'calibration_quality_verdict.json').write_text(json.dumps(verdict,indent=2))

# reliability curves for the focal class of each dataset (one representative model)
fig,axs=plt.subplots(1,3,figsize=(13,4.2))
def rel_plot(ax,P,y,ci,title):
    p=P[:,ci]; t=(y==ci).astype(float); o=np.argsort(p); p,t=p[o],t[o]
    bins=np.array_split(np.arange(len(p)),10); xs=[];ys=[]
    for b in bins:
        if len(b): xs.append(p[b].mean()); ys.append(t[b].mean())
    ax.plot([0,1],[0,1],'k--',lw=0.8); ax.plot(xs,ys,'o-',ms=4)
    ax.set_xlabel('predicted probability'); ax.set_ylabel('empirical frequency'); ax.set_title(title); ax.set_xlim(0,1); ax.set_ylim(0,1)
# NSL R2L
Pn=np.load(config.PROC_DIR/'probs_rf_s42.npz')['S_pool'].astype(np.float64)
rel_plot(axs[0],Pn,y_sp_nsl,{c:i for i,c in enumerate(config.CANONICAL_CLASSES)}['R2L'],'NSL-KDD  R2L (rf)')
# CIC DoS (R1 srcpool)
spx=np.load(config.PROC_DIR/'cic_R1_holdout_Slowhttptest_srcpool_idx.npy'); ysp=lab_cic(spx)
Pc=np.load(config.DATA_DIR/'cic_probs'/'R1_holdout_Slowhttptest__rf__seed42.npz')['srcpool'].astype(np.float64)
rel_plot(axs[1],Pc,ysp,1,'CIC  DoS (rf)')
# UGR nerisbotnet
Pu=np.load(config.DATA_DIR/'ugr16_probs'/'ugr16__rf__seed42.npz')['srcpool'].astype(np.float64)
rel_plot(axs[2],Pu,y_sp_ugr,U2I['nerisbotnet'],'UGR  nerisbotnet (rf)')
fig.suptitle('Reliability of calibrated probabilities (focal class, source pool)'); fig.tight_layout()
fig.savefig(config.REPORTS_DIR/'calibration_reliability.png',dpi=140); print('\nreliability figure saved')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb26: calibration quality - calibrated probs are well-calibrated in-distribution (Brier/ECE/reliability)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


CALIBRATION QUALITY (calibrated probs, source pool, per class):
   dataset       class  brier    ece
cicids2017      Benign 0.0001 0.0001
cicids2017         DoS 0.0001 0.0001
    nslkdd         DoS 0.0003 0.0002
    nslkdd      Normal 0.0018 0.0005
    nslkdd       Probe 0.0007 0.0001
    nslkdd         R2L 0.0007 0.0001
    nslkdd         U2R 0.0002 0.0001
     ugr16  background 0.0158 0.0014
     ugr16         dos 0.0001 0.0001
     ugr16 nerisbotnet 0.0144 0.0011
     ugr16      scan11 0.0171 0.0019
     ugr16      scan44 0.0173 0.0014

per-dataset mean Brier / ECE:
            mean_brier  mean_ece
dataset                         
cicids2017      0.0001    0.0001
nslkdd          0.0007    0.0002
ugr16           0.0130    0.0012

reliability figure saved
